# DATA 612 – Project 4: Accuracy and Beyond

**Author:** Kevin Martin  
**Course:** DATA 612 – Recommender Systems  
**Program:** MS Data Science, CUNY School of Professional Studies  
**Date:** June 2026

## Introduction

In the previous projects, I focused on building recommender systems and evaluating how accurately they predicted user ratings. For this project, I wanted to take the next step by comparing two different recommendation approaches—Item-Based Collaborative Filtering and SVD Matrix Factorization—while also looking beyond prediction accuracy.

One of the ideas introduced in this unit is that the most accurate recommender system is not always the one that provides the best experience for the user. Because of that, I will also explore novelty as a recommendation goal. My goal is to see how balancing accuracy with user experience can lead to recommendations that are not only accurate but also more interesting and useful to the user.


## Project Design Choice

The project instructions provided the option of working in a small group, using a different dataset, or both. For this project, I decided to continue using the synthetic movie ratings dataset from my previous projects.

I made this decision because I wanted to focus on the purpose of this assignment, which is evaluating recommender system algorithms using accuracy and other recommendation metrics. By continuing with the same dataset, I could make direct comparisons between the algorithms under consistent conditions and spend more time exploring how adding novelty affects the recommendations instead of preparing and validating a new dataset. I felt this approach allowed me to better focus on the learning objectives of the assignment while building upon the work completed in the previous projects.




In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error

## Create the Movie Ratings Dataset

Since I decided to continue using the same synthetic movie ratings dataset from the previous projects, the first step was to recreate the dataset. Keeping the same users, movies, and rating structure allows me to make direct comparisons between the recommendation algorithms without introducing differences caused by a new dataset.

The dataset contains 50 users and 20 movies with ratings ranging from 1 to 5. Approximately 15% of the ratings are intentionally replaced with missing values (NaN) to better represent a real-world recommender system where users typically rate only a small portion of the available items.


In [2]:
np.random.seed(42)

users = [f"User_{i}" for i in range(1, 51)]

movies = [
    "The Shawshank Redemption",
    "The Godfather",
    "The Dark Knight",
    "Pulp Fiction",
    "Forrest Gump",
    "The Matrix",
    "Inception",
    "The Lion King",
    "Titanic",
    "Jurassic Park",
    "Toy Story",
    "Finding Nemo",
    "Avengers: Endgame",
    "Goodfellas",
    "Gladiator",
    "The Social Network",
    "La La Land",
    "Get Out",
    "Interstellar",
    "Coco"
]

ratings = pd.DataFrame(
    np.random.randint(1, 6, size=(50, 20)).astype(float),
    index=users,
    columns=movies
)

mask = np.random.rand(50, 20) < 0.15

ratings[mask] = np.nan

ratings.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
User_1,4.0,5.0,3.0,5.0,5.0,2.0,3.0,3.0,3.0,5.0,4.0,3.0,5.0,NaN,4.0,2.0,NaN,5.0,1.0,4.0
User_2,2.0,5.0,NaN,1.0,1.0,NaN,3.0,2.0,4.0,4.0,3.0,4.0,4.0,1.0,3.0,5.0,NaN,5.0,1.0,2.0
User_3,4.0,1.0,4.0,2.0,NaN,1.0,2.0,5.0,NaN,4.0,NaN,4.0,4.0,5.0,NaN,1.0,4.0,2.0,4.0,2.0
User_4,NaN,4.0,5.0,NaN,2.0,4.0,2.0,2.0,4.0,4.0,1.0,5.0,5.0,2.0,5.0,2.0,1.0,4.0,NaN,4.0
User_5,5.0,1.0,NaN,5.0,1.0,1.0,NaN,NaN,4.0,3.0,3.0,1.0,NaN,3.0,1.0,3.0,5.0,2.0,2.0,1.0


## Initial Observations

After creating the dataset, I wanted to take a quick look at the ratings before moving on to the recommendation algorithms. The dataset contains 50 users and 20 movies with ratings ranging from 1 to 5. As expected, some of the ratings are missing because approximately 15% of the values were intentionally replaced with NaN.

Although this is a simulated dataset, the missing ratings represent a common challenge in real-world recommender systems. Most users only rate a small number of available items, so handling missing data is an important step before comparing recommendation algorithms.


## Examine Missing Values

Before filling in the missing ratings, I wanted to see how much of the dataset was actually incomplete. Understanding the amount of missing data helps explain why this step is necessary before applying any recommendation algorithm.

In this dataset, approximately 15% of the ratings are missing. This reflects a situation that is common in real-world recommender systems, where users usually rate only a small number of the available movies. As a result, most recommendation algorithms require some method of handling missing values before meaningful comparisons can be made.


In [3]:
total_cells = ratings.size

missing_values = ratings.isna().sum().sum()

missing_percent = (missing_values / total_cells) * 100

print("Total ratings matrix cells:", total_cells)
print("Missing ratings:", missing_values)
print("Percent missing:", round(missing_percent, 1), "%")

Total ratings matrix cells: 1000
Missing ratings: 159
Percent missing: 15.9 %


### Interpretation

The results confirmed that the dataset contains **159 missing ratings**, or about **15.9%** of the total ratings matrix. Although most of the ratings are available, the missing values still need to be addressed before comparing the recommendation algorithms. Leaving the missing values unchanged could affect how similarities between movies are calculated and how accurately the SVD model identifies hidden patterns. Filling in these missing values helps create a more reliable comparison between the two recommendation approaches.



## Handle Missing Values

Before comparing the recommendation algorithms, I needed to decide how to handle the missing ratings in the dataset. There are several ways this can be done, but I decided to replace the missing values with each movie's average rating.

I felt this was the best approach because a missing rating does not necessarily mean the user disliked the movie. In many cases, it simply means the user never watched the movie or chose not to rate it. Using the movie average provides a reasonable estimate while preserving the overall rating pattern for each movie and creating a complete dataset for both recommendation algorithms.


In [4]:
movie_means = ratings.mean()

ratings_filled = ratings.fillna(movie_means)

ratings_filled.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
User_1,4.000000,5.0,3.000000,5.000000,5.000000,2.0,3.000000,3.000000,3.0,5.0,4.000000,3.0,5.00000,3.022222,4.000000,2.0,2.837209,5.0,1.000,4.0
User_2,2.000000,5.0,2.974359,1.000000,1.000000,2.7,3.000000,2.000000,4.0,4.0,3.000000,4.0,4.00000,1.000000,3.000000,5.0,2.837209,5.0,1.000,2.0
User_3,4.000000,1.0,4.000000,2.000000,2.767442,1.0,2.000000,5.000000,3.0,4.0,2.609756,4.0,4.00000,5.000000,2.931818,1.0,4.000000,2.0,4.000,2.0
User_4,2.804878,4.0,5.000000,3.219512,2.000000,4.0,2.000000,2.000000,4.0,4.0,1.000000,5.0,5.00000,2.000000,5.000000,2.0,1.000000,4.0,2.925,4.0
User_5,5.000000,1.0,2.974359,5.000000,1.000000,1.0,3.170732,3.166667,4.0,3.0,3.000000,1.0,3.27907,3.000000,1.000000,3.0,5.000000,2.0,2.000,1.0


### Interpretation

Looking at the completed ratings matrix, replacing the missing values with each movie's average rating gave me a complete dataset to work with while keeping the overall rating patterns for each movie. I felt this was a better approach than replacing the missing values with zero because a missing rating does not necessarily mean the user disliked the movie. In many cases, it simply means they never watched it or chose not to rate it. Using the movie average provided a reasonable estimate that allowed me to continue comparing the recommendation algorithms without introducing unnecessary bias into the data.


### Why This Step Matters

Since the goal of this project is to compare two recommender system algorithms, I wanted both algorithms to start with the same completed dataset. That way, if one algorithm performs better than the other, I know the difference is coming from the algorithm itself and not from how the data was prepared. Using the same dataset creates a fair comparison and makes the results easier to interpret.


## Item-Based Collaborative Filtering

The first recommendation approach I wanted to evaluate was Item-Based Collaborative Filtering. Instead of looking for users with similar preferences, this method compares movies based on how users rated them. If two movies receive similar ratings from many users, the algorithm considers them to be similar and uses those relationships to generate recommendations.

My goal is to use this approach as a baseline before comparing it with SVD Matrix Factorization later in the project.


## Calculate Item Similarity

Now that the dataset is complete, I can start building the first recommendation model. The next step is to see which movies are most similar based on how users rated them.

To do this, I will use cosine similarity. Since I want to compare movies instead of users, I first transpose the ratings matrix so that each movie can be compared with every other movie. These similarity scores will be used to help generate movie recommendations.


In [5]:
item_similarity = cosine_similarity(ratings_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=ratings_filled.columns,
    columns=ratings_filled.columns
)

item_similarity_df.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
The Shawshank Redemption,1.000000,0.849656,0.822245,0.866608,0.867901,0.813441,0.882215,0.858685,0.860021,0.870529,0.849873,0.822950,0.878394,0.834820,0.773191,0.817580,0.854932,0.842480,0.839638,0.857815
The Godfather,0.849656,1.000000,0.824720,0.861330,0.854592,0.885723,0.868425,0.852500,0.828910,0.845070,0.856740,0.845194,0.855076,0.806817,0.848335,0.817531,0.844704,0.847765,0.829560,0.843038
The Dark Knight,0.822245,0.824720,1.000000,0.864935,0.812034,0.823084,0.868272,0.847168,0.836759,0.835386,0.826192,0.796930,0.818470,0.875568,0.844225,0.776230,0.763453,0.824566,0.796690,0.857992
Pulp Fiction,0.866608,0.861330,0.864935,1.000000,0.823757,0.837416,0.847695,0.884064,0.845369,0.868000,0.891340,0.817531,0.859120,0.851632,0.834981,0.840734,0.847140,0.845707,0.831106,0.873908
Forrest Gump,0.867901,0.854592,0.812034,0.823757,1.000000,0.830936,0.878198,0.807695,0.811518,0.855797,0.817890,0.815420,0.871210,0.824284,0.824972,0.801948,0.836017,0.830669,0.823874,0.852286


### Interpretation

Looking at the similarity matrix, I can see how closely the movies are related based on the way users rated them. Movies with similarity scores closer to 1 have more similar rating patterns, while lower scores indicate that users tended to rate the movies differently.

At this point, I am not making recommendations yet. My goal here is simply to identify the relationships between movies. Those relationships will be used in the next step to generate recommendations.

### Why This Step Matters

Before I can recommend similar movies, the recommender system first needs to understand which movies are actually related based on user ratings. Building the similarity matrix creates those relationships and gives me a foundation for generating recommendations. It also gives me a baseline that I can later compare with the SVD recommendation model.
